# Predial Vision MX - Detección de Edificios en Nicolás Romero

Entrenamiento de modelo de segmentación semántica para detectar edificios usando:
- **Imagery:** ESRI World Imagery (~1m/pixel) 
- **Labels:** OSM Building Footprints (1,707 polígonos)
- **Modelo:** DeepLabV3+ con MobileNetV2 backbone
- **Framework:** PyTorch + segmentation_models_pytorch

**Repo:** github.com/MarxCha/predial-vision-mx

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

In [ ]:
!pip install -q segmentation-models-pytorch rasterio

## 1. Cargar datos del dataset

In [ ]:
import os
import json
import numpy as np
import rasterio
from rasterio.features import rasterize
from shapely.geometry import shape

# Paths - ajustar según Kaggle dataset
DATA_DIR = '/kaggle/input/predial-vision-nr-data'
OUT_DIR = '/kaggle/working'

# Verificar archivos
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024 / 1024
        print(f'  {os.path.relpath(path, DATA_DIR):50s} {size:.1f} MB')

In [ ]:
# Cargar imagery
imagery_path = os.path.join(DATA_DIR, 'nicolas-romero', 'nicolas-romero.tif')
with rasterio.open(imagery_path) as src:
    imagery = src.read()  # (3, H, W)
    transform = src.transform
    crs = src.crs
    profile = src.profile.copy()
    
print(f'Imagery: {imagery.shape} dtype={imagery.dtype}')
print(f'CRS: {crs}')
print(f'Bounds: {src.bounds}')

In [ ]:
# Cargar buildings GeoJSON y rasterizar como máscara
buildings_path = os.path.join(DATA_DIR, 'nicolas-romero', 'buildings.geojson')
with open(buildings_path) as f:
    buildings = json.load(f)

print(f'Buildings: {len(buildings["features"])} polígonos')

# Rasterizar buildings sobre la misma grilla que la imagery
geometries = [(shape(feat['geometry']), 1) for feat in buildings['features']]

with rasterio.open(imagery_path) as src:
    mask = rasterize(
        geometries,
        out_shape=(src.height, src.width),
        transform=src.transform,
        fill=0,
        dtype='uint8'
    )

print(f'Mask: {mask.shape}')
print(f'Building pixels: {np.sum(mask == 1):,} ({100*np.sum(mask==1)/mask.size:.2f}%)')
print(f'Background pixels: {np.sum(mask == 0):,} ({100*np.sum(mask==0)/mask.size:.2f}%)')

## 2. Crear chips de entrenamiento

In [ ]:
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

CHIP_SIZE = 256
STRIDE = 128

class BuildingChipDataset(Dataset):
    def __init__(self, imagery, mask, chip_size, stride, transform=None, min_building_pct=0.001):
        """Extract chips from imagery and mask."""
        self.chips = []
        self.labels = []
        self.transform = transform
        
        C, H, W = imagery.shape
        for y in range(0, H - chip_size, stride):
            for x in range(0, W - chip_size, stride):
                img_chip = imagery[:, y:y+chip_size, x:x+chip_size]
                msk_chip = mask[y:y+chip_size, x:x+chip_size]
                
                # Skip chips that are all black (NODATA)
                if img_chip.mean() < 5:
                    continue
                
                self.chips.append(img_chip)
                self.labels.append(msk_chip)
        
        print(f'Total chips: {len(self.chips)}')
        building_chips = sum(1 for m in self.labels if m.sum() > 0)
        print(f'Chips with buildings: {building_chips} ({100*building_chips/len(self.chips):.1f}%)')
    
    def __len__(self):
        return len(self.chips)
    
    def __getitem__(self, idx):
        img = self.chips[idx].transpose(1, 2, 0).astype(np.float32)  # (H, W, C)
        msk = self.labels[idx].astype(np.float32)
        
        if self.transform:
            augmented = self.transform(image=img, mask=msk)
            img = augmented['image']
            msk = augmented['mask']
        else:
            img = torch.from_numpy(img.transpose(2, 0, 1))  # (C, H, W)
            msk = torch.from_numpy(msk)
        
        return img / 255.0, msk.long()

# Augmentaciones para training
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    ToTensorV2(),
])

val_transform = A.Compose([
    ToTensorV2(),
])

# Split: 80% train, 20% val (spatial split by rows)
split_row = int(imagery.shape[1] * 0.8)

print('=== Training set ===')
train_dataset = BuildingChipDataset(
    imagery[:, :split_row, :], mask[:split_row, :],
    CHIP_SIZE, STRIDE, train_transform
)

print('\n=== Validation set ===')
val_dataset = BuildingChipDataset(
    imagery[:, split_row:, :], mask[split_row:, :],
    CHIP_SIZE, CHIP_SIZE, val_transform  # no overlap for val
)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 3. Modelo DeepLabV3+ con MobileNetV2

In [ ]:
import segmentation_models_pytorch as smp

model = smp.DeepLabV3Plus(
    encoder_name='mobilenet_v2',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,  # binary: building vs background
    activation=None,  # raw logits
).to(DEVICE)

# Loss y optimizer
criterion = smp.losses.DiceLoss(mode='binary') + smp.losses.SoftBCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: DeepLabV3+ MobileNetV2')
print(f'Parameters: {total_params:,}')

## 4. Entrenamiento

In [ ]:
from tqdm import tqdm

NUM_EPOCHS = 30
best_val_f1 = 0
history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_iou': []}

for epoch in range(NUM_EPOCHS):
    # === Training ===
    model.train()
    train_loss = 0
    for imgs, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', leave=False):
        imgs = imgs.float().to(DEVICE)
        masks = masks.float().to(DEVICE).unsqueeze(1)
        
        preds = model(imgs)
        loss = criterion(preds, masks)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    scheduler.step()
    
    # === Validation ===
    model.eval()
    val_loss = 0
    tp, fp, fn = 0, 0, 0
    
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.float().to(DEVICE)
            masks = masks.float().to(DEVICE).unsqueeze(1)
            
            preds = model(imgs)
            val_loss += criterion(preds, masks).item()
            
            pred_binary = (torch.sigmoid(preds) > 0.5).float()
            tp += (pred_binary * masks).sum().item()
            fp += (pred_binary * (1 - masks)).sum().item()
            fn += ((1 - pred_binary) * masks).sum().item()
    
    val_loss /= len(val_loader)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(f1)
    history['val_iou'].append(iou)
    
    print(f'Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
          f'F1: {f1:.4f} | IoU: {iou:.4f} | Prec: {precision:.4f} | Rec: {recall:.4f}')
    
    # Guardar mejor modelo
    if f1 > best_val_f1:
        best_val_f1 = f1
        torch.save(model.state_dict(), f'{OUT_DIR}/best_model.pth')
        print(f'  -> Mejor modelo guardado (F1={f1:.4f})')

print(f'\nMejor F1: {best_val_f1:.4f}')

In [ ]:
# Gráficas de entrenamiento
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history['val_f1'])
axes[1].set_title(f'F1 Score (best={best_val_f1:.4f})')

axes[2].plot(history['val_iou'])
axes[2].set_title('IoU')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=150)
plt.show()

## 5. Predicciones sobre toda la imagen

In [ ]:
# Cargar mejor modelo
model.load_state_dict(torch.load(f'{OUT_DIR}/best_model.pth'))
model.eval()

C, H, W = imagery.shape
pred_mask = np.zeros((H, W), dtype=np.float32)
count_mask = np.zeros((H, W), dtype=np.float32)

# Sliding window prediction
PRED_CHIP = 256
PRED_STRIDE = 128

with torch.no_grad():
    total = ((H - PRED_CHIP) // PRED_STRIDE + 1) * ((W - PRED_CHIP) // PRED_STRIDE + 1)
    done = 0
    for y in range(0, H - PRED_CHIP, PRED_STRIDE):
        batch_imgs = []
        batch_coords = []
        
        for x in range(0, W - PRED_CHIP, PRED_STRIDE):
            chip = imagery[:, y:y+PRED_CHIP, x:x+PRED_CHIP].astype(np.float32) / 255.0
            if chip.mean() < 0.02:  # skip NODATA
                continue
            batch_imgs.append(chip)
            batch_coords.append((y, x))
            
            if len(batch_imgs) == 32:
                batch = torch.from_numpy(np.stack(batch_imgs)).to(DEVICE)
                preds = torch.sigmoid(model(batch)).cpu().numpy()[:, 0]
                for pred, (cy, cx) in zip(preds, batch_coords):
                    pred_mask[cy:cy+PRED_CHIP, cx:cx+PRED_CHIP] += pred
                    count_mask[cy:cy+PRED_CHIP, cx:cx+PRED_CHIP] += 1
                batch_imgs = []
                batch_coords = []
        
        # Process remaining
        if batch_imgs:
            batch = torch.from_numpy(np.stack(batch_imgs)).to(DEVICE)
            preds = torch.sigmoid(model(batch)).cpu().numpy()[:, 0]
            for pred, (cy, cx) in zip(preds, batch_coords):
                pred_mask[cy:cy+PRED_CHIP, cx:cx+PRED_CHIP] += pred
                count_mask[cy:cy+PRED_CHIP, cx:cx+PRED_CHIP] += 1
        
        done += (W - PRED_CHIP) // PRED_STRIDE + 1
        if y % (PRED_STRIDE * 10) == 0:
            print(f'  {100*done/total:.0f}%')

# Average overlapping predictions
count_mask[count_mask == 0] = 1
pred_mask = pred_mask / count_mask
pred_binary = (pred_mask > 0.5).astype(np.uint8)

print(f'\nPredicciones:')
print(f'  Building: {np.sum(pred_binary == 1):,} pixels ({100*np.sum(pred_binary==1)/pred_binary.size:.2f}%)')
print(f'  Background: {np.sum(pred_binary == 0):,} pixels')

In [ ]:
# Guardar predicciones como GeoTIFF
pred_profile = profile.copy()
pred_profile.update(dtype='uint8', count=1, compress='lzw')

pred_tif_path = f'{OUT_DIR}/nicolas_romero_predictions.tif'
with rasterio.open(pred_tif_path, 'w', **pred_profile) as dst:
    dst.write(pred_binary[np.newaxis, :, :])

print(f'Predicciones guardadas: {pred_tif_path}')
print(f'Tamaño: {os.path.getsize(pred_tif_path)/1024/1024:.1f} MB')

In [ ]:
# Vectorizar predicciones a GeoJSON (polígonos de edificios)
from rasterio.features import shapes as rasterio_shapes
from shapely.geometry import shape as shapely_shape, mapping

print('Vectorizando predicciones...')
polygons = []
with rasterio.open(pred_tif_path) as src:
    pred_data = src.read(1)
    for geom, value in rasterio_shapes(pred_data, transform=src.transform):
        if value == 1:  # Building
            poly = shapely_shape(geom)
            if poly.area > 1e-9:  # filtrar polígonos tiny
                polygons.append({
                    'type': 'Feature',
                    'properties': {'class': 'building', 'area_m2': poly.area * 111000**2},
                    'geometry': mapping(poly)
                })

geojson = {'type': 'FeatureCollection', 'features': polygons}
geojson_path = f'{OUT_DIR}/nicolas_romero_buildings_predicted.geojson'
with open(geojson_path, 'w') as f:
    json.dump(geojson, f)

print(f'Polígonos detectados: {len(polygons)}')
print(f'GeoJSON guardado: {geojson_path}')

## 6. Visualización

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

# Imagery
axes[0].imshow(imagery.transpose(1, 2, 0))
axes[0].set_title('Imagery Satelital - Nicolás Romero')
axes[0].axis('off')

# Ground truth
axes[1].imshow(imagery.transpose(1, 2, 0))
axes[1].imshow(mask, cmap='Oranges', alpha=0.5)
axes[1].set_title(f'Labels OSM ({len(buildings["features"])} buildings)')
axes[1].axis('off')

# Predictions
axes[2].imshow(imagery.transpose(1, 2, 0))
axes[2].imshow(pred_binary, cmap='Oranges', alpha=0.5)
axes[2].set_title(f'Predicciones ({len(polygons)} polígonos)')
axes[2].axis('off')

plt.suptitle('Predial Vision MX - Detección de Edificios - Nicolás Romero, Edo. de México', fontsize=16)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/resultados_nicolas_romero.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Métricas y resumen

In [ ]:
# Métricas finales sobre toda la imagen
tp = np.sum((pred_binary == 1) & (mask == 1))
fp = np.sum((pred_binary == 1) & (mask == 0))
fn = np.sum((pred_binary == 0) & (mask == 1))
tn = np.sum((pred_binary == 0) & (mask == 0))

precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)
iou = tp / (tp + fp + fn + 1e-8)

print('=' * 50)
print('PREDIAL VISION MX - MÉTRICAS FINALES')
print('=' * 50)
print(f'Nicolás Romero, Estado de México')
print(f'')
print(f'Precision:  {precision:.4f}')
print(f'Recall:     {recall:.4f}')
print(f'F1 Score:   {f1:.4f}')
print(f'IoU:        {iou:.4f}')
print(f'')
print(f'Polígonos detectados: {len(polygons)}')
print(f'Modelo: DeepLabV3+ MobileNetV2')
print(f'Epochs: {NUM_EPOCHS}')
print(f'Mejor val F1: {best_val_f1:.4f}')
print(f'')
print(f'Archivos generados:')
print(f'  nicolas_romero_predictions.tif (raster)')
print(f'  nicolas_romero_buildings_predicted.geojson (vectorial)')
print(f'  best_model.pth (modelo PyTorch)')
print(f'  training_curves.png')
print(f'  resultados_nicolas_romero.png')

# Guardar métricas
eval_results = {
    'scene': 'nicolas_romero',
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1),
    'iou': float(iou),
    'polygons_detected': len(polygons),
    'model': 'DeepLabV3+ MobileNetV2',
    'epochs': NUM_EPOCHS,
    'best_val_f1': float(best_val_f1)
}
with open(f'{OUT_DIR}/eval.json', 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f'\neval.json guardado')